# Single-fiber DM piston scans across mount focus

Analysis only: this notebook never imports acquisition code or connects to hardware.
The companion terminal script saves one spectrum per `[focus, segment, piston]`.
Spectra already average frames, subtract dark and **sum rows**; broadband flux is
`SPECTRA.sum(axis=-1)` in ADU/frame. OPD is **2 x absolute commanded DM stroke**, in um.
Focus is absolute mount Z in controller pulses, not nanometres.

The numbered 37-hexagon layout and sine-squared fringe model follow the supplied
`null_scan_parasitic_interference_analysis.ipynb`. Here there is one dispersed output:
one visibility-coloured DM map per focus, plus an MP4 with the same colour scale.
The segment inspector shows focus versus OPD, a selected fringe fit, and visibility
versus focus. Missing/rejected fits are grey; a sufficiently flat positive trace has V=0.

Run the acquisition from the repository root:
```powershell
python Matteo/Fiber_DM_piston_scans/fiber_DMfocus_piston_scan.py
```
Set `SEGMENT_PTT`, `PISTON_START/END/STEP`, `NFRAMES` and `FOCUS_START/END/STEP`
in that script. Unset focus values prompt in the terminal. Both grid endpoints
are included; the interval must be an integer multiple of the requested step.
Check the plan with `--check-only --focus-start <Z0> --focus-end <Z1> --focus-step <dZ>`.
Outputs go to the configured alignment root under
`YYYYMMDD/FIber_DMfocus_piston_scan/fiber_dm_piston_N.fits` plus a JSON sidecar.

Select that FITS in the first code cell here and run all cells. Analysis outputs
include fit results CSV, settings JSON, one DM PNG per focus, 37 per-segment
focus/OPD diagnostic PNGs, and `visibility_vs_focus.mp4`.

The notebook uses the same Windows `OUTPUT_ROOT` as the fiber-focus and fiber-piston
notebooks. It reads only `OUTPUT_ROOT/YYYYMMDD/FIber_DMfocus_piston_scan/`.
For example: `OUTPUT_ROOT/20261124/FIber_DMfocus_piston_scan/fiber_dm_piston_1.fits`.
Set `SCAN_DATE="20261124"` to select that day's folder, or leave it `None` to choose
the newest scan across dates. `DATA_FILE` can explicitly select a full path or a path
relative to `OUTPUT_ROOT`. On the instrument, acquisition uses the same configured
alignment data root as the existing focus/piston scripts, normally
`/home/scexao/glint/glintdata/alignment_scan`.


In [ ]:
# USER CONTROLS - edit this standalone cell first
from pathlib import Path

# Same Windows data-copy root as the fiber-focus and fiber-piston notebooks.
OUTPUT_ROOT = Path(r"C:\Users\mcol0021\OneDrive - The University of Sydney (Staff)\Desktop\GLINT_test reports\Fiber_scans")
SCAN_DATE = None  # e.g. "20261124"; None searches all date folders
SCAN_FOLDER = "FIber_DMfocus_piston_scan"
DATA_FILE = None  # None selects newest scan; or full path / path relative to OUTPUT_ROOT
USE_SYNTHETIC_DEMO = False  # explicitly enable to preview without acquired data
OUTPUT_DIR = None  # None -> beside DATA_FILE / <scan_name>_analysis

# Fit model: F(OPD) = c0 - A*sin(k*(OPD-mean(OPD)) + phi)**2
# V = (Fmax-Fmin)/(Fmax+Fmin) = A/(2*c0-A).
OPD_PERIOD_BOUNDS_UM = None  # None: reference notebook range [OPD_span/1.8, OPD_span/0.6]
# Can instead constrain the calibrated fringe period, e.g. (1.2, 2.0) in OPD um.
N_FREQUENCY_TRIALS = 400
MIN_POINTS_FOR_FIT = 8
FLAT_FRACTION_THRESHOLD = 1e-3
MIN_FIT_R2 = 0.35
MAX_VISIBILITY = 1.0

VISIBILITY_CMAP = "inferno"
VISIBILITY_LIMITS = (0.0, 1.0)  # shared by every focus / movie frame
VISIBILITY_LOG_SCALE = False  # if True, requires positive lower limit, e.g. (0.005, 1.0)
SELECTED_SEGMENT = 0
SELECTED_FOCUS_INDEX = 0
NORMALIZE_FOCUS_OPD_ROWS = False  # divide each focus's flux trace by its mean for display only

SAVE_FOCUS_PNGS = True
SAVE_SEGMENT_PNGS = True  # exports all 37 focus-versus-OPD + visibility diagnostic figures
EXPORT_MP4 = True
MOVIE_FPS = 2
MOVIE_HOLD_FRAMES = 2  # repeated frames per focus; focus order follows acquisition


In [ ]:
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize, LogNorm
from matplotlib.animation import FFMpegWriter
from IPython.display import display, Video


def find_scan():
    root = Path(OUTPUT_ROOT).expanduser()
    date_folder = "*" if SCAN_DATE is None else str(SCAN_DATE)
    if SCAN_DATE is not None and (len(date_folder) != 8 or not date_folder.isdigit()):
        raise ValueError("SCAN_DATE must be None or a YYYYMMDD date, e.g. '20261124'")
    pattern = f"{date_folder}/{SCAN_FOLDER}/fiber_dm_piston_*.fits"
    print("Acquisition data pattern:", root/pattern)
    files = list(root.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No scan FITS matching {root/pattern}. "
                                "Set OUTPUT_ROOT to the common fiber-scan data root, SCAN_DATE to a date, "
                                "or DATA_FILE to a specific FITS. Synthetic previews require USE_SYNTHETIC_DEMO=True.")
    return max(files, key=lambda p: p.stat().st_mtime)


def selected_scan_path():
    if DATA_FILE is None:
        return find_scan().resolve()
    path = Path(DATA_FILE).expanduser()
    return (path if path.is_absolute() else Path(OUTPUT_ROOT).expanduser()/path).resolve()


def load_scan(path):
    with fits.open(path, memmap=False, checksum=True) as hdul:
        if hdul[0].header.get("SCANTYPE") != "FIBER_DM_PISTON":
            raise ValueError("Select a FITS from the new all-segment DM/focus acquisition script")
        meta = json.loads(hdul["SCAN_SETTINGS"].data["JSON"][0])
        data = {key.lower(): np.array(hdul[key].data, copy=True)
                for key in ("SPECTRA", "ACQUIRED", "FOCUS_CMD", "PISTON_CMD", "OPD", "SEGMENT", "SPECTRAL_X")}
        saved_flux = np.array(hdul["FLUX"].data, dtype=float)
    expected = (len(data["focus_cmd"]), 37, len(data["piston_cmd"]), len(data["spectral_x"]))
    if data["spectra"].shape != expected or data["acquired"].shape != expected[:-1]:
        raise ValueError(f"Unexpected cube / mask shape; expected {expected}")
    if not np.array_equal(data["segment"], np.arange(37)):
        raise ValueError("Expected segments 0..36 in order")
    if not np.allclose(data["opd"], 2*data["piston_cmd"]):
        raise ValueError("OPD is not twice commanded DM stroke")
    data["acquired"] = data["acquired"].astype(bool)
    data["flux"] = data["spectra"].sum(axis=-1)
    if not np.allclose(data["flux"][data["acquired"]], saved_flux[data["acquired"]]):
        raise ValueError("Stored FLUX and integrated spectra disagree")
    data["flux"][~data["acquired"]] = np.nan
    data["metadata"] = meta
    return data


def synthetic_demo():
    rng = np.random.default_rng(7)
    z = np.arange(1500, 2001, 100, dtype=float)
    p = np.linspace(-0.6, 0.3, 19)
    opd = 2*p
    v = 0.03 + 0.55*np.exp(-((z[:, None]-1750)/220)**2)*(0.3+0.7*np.arange(37)[None, :]/36)
    phase = np.arange(37)*0.15
    flux = 5000*(1+v[:, :, None]*np.cos(2*np.pi*opd[None, None, :]/1.6+phase[None, :, None]))
    flux += rng.normal(0, 2, flux.shape)
    profile = np.exp(-0.5*((np.arange(109)-54)/35)**2)
    spectra = flux[..., None]*profile/profile.sum()
    return dict(spectra=spectra, flux=flux, acquired=np.ones(flux.shape, dtype=bool),
                focus_cmd=z, piston_cmd=p, opd=opd, segment=np.arange(37),
                spectral_x=np.arange(158, 267), metadata={"status":"SYNTHETIC DEMO", "nframes":25})


if USE_SYNTHETIC_DEMO:
    scan = synthetic_demo()
    source_path = None
    source_label = "SYNTHETIC DEMO - not instrument measurements"
    output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else Path.cwd()/"synthetic_dm_piston_analysis"
else:
    source_path = selected_scan_path()
    scan = load_scan(source_path)
    source_label = source_path.name
    output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else source_path.parent/(source_path.stem+"_analysis")
output_dir.mkdir(parents=True, exist_ok=True)
focus = scan["focus_cmd"]
opd = scan["opd"]
flux = scan["flux"]
print(source_label)
print("Cube [focus,segment,piston,column]:", scan["spectra"].shape)
print("Completed:", scan["acquired"].sum(), "/", scan["acquired"].size, "status:", scan["metadata"]["status"])
print("Outputs:", output_dir.resolve())


## Fringe fitting
For each focus and segment, fit the integrated 109-column spectrum versus OPD using
`F = m + B*cos(2*k*x) + C*sin(2*k*x)`, equivalent to the reference sine-squared dip.
Then `c0=m+hypot(B,C)`, `A=2*hypot(B,C)` and `V=hypot(B,C)/m`.
A frequency grid avoids sensitivity to nonlinear initial guesses. Fit quality, residual
RMS and frequency-bound flags are exported. Rejected/insufficient fits retain NaN visibility;
flat positive traces have V=0. The contrast is model-based; it is not a noise-bias-corrected
visibility or a calibrated uncertainty estimate. Narrow OPD coverage can leave period and
contrast weakly constrained; inspect residuals and bound flags before interpreting a map.


In [ ]:
def fit_fringe(x, y):
    good = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[good], np.asarray(y)[good]
    result = dict(visibility=np.nan, visibility_raw=np.nan, mean_flux=float(y.mean()) if y.size else np.nan,
                  c0=np.nan, A=np.nan, k=np.nan, phi=np.nan, x_center=np.nan,
                  r2=np.nan, residual_rms=np.nan, period_um=np.nan, n_points=len(x),
                  fit_success=False, frequency_at_bound=False, fit_reason="too_few_points")
    if len(x) < MIN_POINTS_FOR_FIT or np.ptp(x) <= 0:
        return result
    if y.mean() <= 0:
        result["fit_reason"] = "nonpositive_mean_flux"
        return result
    if np.ptp(y)/abs(y.mean()) < FLAT_FRACTION_THRESHOLD:
        result.update(visibility=0., visibility_raw=0., c0=float(y.mean()), A=0.,
                      residual_rms=float(np.std(y)), fit_reason="flat_below_threshold")
        return result
    period_bounds = OPD_PERIOD_BOUNDS_UM or (np.ptp(x)/1.8, np.ptp(x)/0.6)
    lo, hi = map(float, period_bounds)
    if not 0 < lo < hi or not np.all(np.isfinite([lo, hi])):
        raise ValueError("OPD_PERIOD_BOUNDS_UM must be two finite positive increasing values")
    if N_FREQUENCY_TRIALS < 3:
        raise ValueError("N_FREQUENCY_TRIALS must be at least 3")
    x0 = x-x.mean()
    k_grid = np.linspace(np.pi/hi, np.pi/lo, N_FREQUENCY_TRIALS)
    best = None
    for index, k in enumerate(k_grid):
        X = np.column_stack([np.ones_like(x0), np.cos(2*k*x0), np.sin(2*k*x0)])
        beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
        if rank != 3:
            continue
        rss = float(np.sum((y-X@beta)**2))
        if best is None or rss < best[0]:
            best = (rss, index, k, beta)
    if best is None:
        result["fit_reason"] = "rank_deficient"
        return result
    rss, index, k, (mean, B, C) = best
    amplitude = float(np.hypot(B, C))
    visibility = float(amplitude/mean) if mean > 0 else np.nan
    r2 = 1-rss/np.sum((y-y.mean())**2)
    result.update(visibility_raw=visibility, c0=float(mean+amplitude), A=2*amplitude,
                  k=float(k), phi=float(-0.5*np.arctan2(C, B)), x_center=float(x.mean()),
                  r2=float(r2), residual_rms=float(np.sqrt(rss/len(y))), period_um=float(np.pi/k),
                  frequency_at_bound=index in (0, len(k_grid)-1))
    if not np.isfinite(visibility) or visibility < 0 or visibility > MAX_VISIBILITY or mean-amplitude < 0:
        result["fit_reason"] = "unphysical_contrast"
    elif r2 < MIN_FIT_R2:
        result["fit_reason"] = "low_r2"
    else:
        result.update(visibility=visibility, fit_success=True, fit_reason="accepted")
    return result


def evaluate_fit(result, x):
    if result["fit_reason"] == "flat_below_threshold":
        return np.full_like(x, result["mean_flux"], dtype=float)
    if not np.isfinite(result["k"]):
        return np.full_like(x, np.nan, dtype=float)
    return result["c0"]-result["A"]*np.sin(result["k"]*(x-result["x_center"])+result["phi"])**2


fit_results = {}
records = []
for iz, z in enumerate(focus):
    for segment in range(37):
        result = fit_fringe(opd, flux[iz, segment])
        fit_results[iz, segment] = result
        records.append(dict(focus_index=iz, focus_z_pulses=z, segment=segment, **result))
results = pd.DataFrame(records)
results.to_csv(output_dir/"visibility_fit_results.csv", index=False)
settings = {name: globals()[name] for name in (
    "OPD_PERIOD_BOUNDS_UM", "N_FREQUENCY_TRIALS", "MIN_POINTS_FOR_FIT", "FLAT_FRACTION_THRESHOLD",
    "MIN_FIT_R2", "MAX_VISIBILITY", "VISIBILITY_CMAP", "VISIBILITY_LIMITS", "VISIBILITY_LOG_SCALE")}
(output_dir/"analysis_settings.json").write_text(json.dumps(dict(source=source_label, settings=settings), indent=2), encoding="utf-8")
visibility = results["visibility"].to_numpy().reshape(len(focus), 37)
print(results.groupby("fit_reason").size().to_string())
display(results.head())


## One DM visibility map per focus
Numbering follows the reference: columns of 4, 5, 6, 7, 6, 5, 4 segments,
top-to-bottom within each column, proceeding right-to-left. Segment labels identify
the piston being swept; all maps use the single saved dispersed fiber output.


In [ ]:
def dm_geometry():
    radius = 1/np.sqrt(3)
    centers = []
    patches = []
    for column, count in enumerate([4, 5, 6, 7, 6, 5, 4]):
        for y in np.arange(count)[::-1]-(count-1)/2:
            center = np.array([(3-column)*1.5*radius, y])
            angles = np.arange(6)*np.pi/3
            vertices = center + radius*np.column_stack([np.cos(angles), np.sin(angles)])
            centers.append(center)
            patches.append(Polygon(vertices, closed=True))
    return np.array(centers), patches


def draw_dm_map(iz):
    centers, patches = dm_geometry()
    cmap = plt.get_cmap(VISIBILITY_CMAP).copy()
    cmap.set_bad("lightgrey")
    if VISIBILITY_LOG_SCALE:
        if VISIBILITY_LIMITS[0] <= 0:
            raise ValueError("Log visibility scale needs a positive lower limit")
        norm = LogNorm(*VISIBILITY_LIMITS)
        values = np.where(np.isfinite(visibility[iz]), np.maximum(visibility[iz], VISIBILITY_LIMITS[0]), np.nan)
    else:
        norm = Normalize(*VISIBILITY_LIMITS)
        values = visibility[iz]
    fig, ax = plt.subplots(figsize=(7.2, 7.0))
    collection = PatchCollection(patches, cmap=cmap, norm=norm, edgecolor="black", linewidth=.8)
    collection.set_array(np.ma.masked_invalid(values))
    ax.add_collection(collection)
    for segment, (x, y) in enumerate(centers):
        ax.text(x, y, str(segment), ha="center", va="center", fontsize=9, color="white",
                bbox=dict(facecolor="black", alpha=.35, edgecolor="none", pad=.6))
    ax.set(xlim=(-3.6, 3.6), ylim=(-4, 4), aspect="equal")
    ax.set_axis_off()
    ax.set_title(f"Single-fiber fringe visibility - Z = {focus[iz]:g} pulses\n"
                 f"{np.isfinite(visibility[iz]).sum()}/37 valid; grey = missing/rejected", fontsize=12)
    fig.colorbar(collection, ax=ax, shrink=.8, label="Visibility (Fmax - Fmin) / (Fmax + Fmin)")
    fig.text(.5, .02, source_label, ha="center", fontsize=8)
    fig.subplots_adjust(bottom=.09, top=.88)
    return fig


for iz in range(len(focus)):
    fig = draw_dm_map(iz)
    if SAVE_FOCUS_PNGS:
        fig.savefig(output_dir/f"visibility_focus_{iz:03d}_z{focus[iz]:g}.png", dpi=160)
    display(fig)
    plt.close(fig)


## Per-segment focus / OPD inspector
Use the dropdowns below, or call `inspect_segment(segment=12, focus_index=2)` directly.
The first panel is measured broadband flux versus **focus and OPD**; the second is the
selected fringe with the sine-squared fit and residuals; the third is visibility versus
focus. Optional row normalization affects only the first panel, never the fitted data.
Unacquired samples remain gaps. Rejected candidate fits are dashed and retain their
reason; CSV stores the raw candidate contrast separately from accepted visibility.


In [ ]:
def cell_edges(values, singleton_width=1.):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return np.array([values[0]-singleton_width/2, values[0]+singleton_width/2])
    mid = (values[:-1]+values[1:])/2
    return np.r_[values[0]-(mid[0]-values[0]), mid, values[-1]+(values[-1]-mid[-1])]


def plot_segment_diagnostics(segment=SELECTED_SEGMENT, focus_index=SELECTED_FOCUS_INDEX,
                             normalize_rows=NORMALIZE_FOCUS_OPD_ROWS):
    segment, focus_index = int(segment), int(focus_index)
    if not 0 <= segment < 37 or not 0 <= focus_index < len(focus):
        raise ValueError("Invalid segment ID or focus index")
    fig = plt.figure(figsize=(16, 6.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 3, height_ratios=(3, 1))
    ax_heat = fig.add_subplot(grid[:, 0])
    ax_trace = fig.add_subplot(grid[0, 1])
    ax_resid = fig.add_subplot(grid[1, 1], sharex=ax_trace)
    ax_v = fig.add_subplot(grid[:, 2])
    displayed = flux[:, segment].copy()
    if normalize_rows:
        count = np.isfinite(displayed).sum(axis=1, keepdims=True)
        means = np.divide(np.nansum(displayed, axis=1, keepdims=True), count,
                          out=np.full((len(focus), 1), np.nan), where=count > 0)
        displayed = np.divide(displayed, means, out=np.full_like(displayed, np.nan), where=means != 0)
    # Sort display coordinates only; result indices always follow acquisition order.
    iz_sort, ip_sort = np.argsort(focus), np.argsort(opd)
    image = ax_heat.pcolormesh(cell_edges(opd[ip_sort], .1), cell_edges(focus[iz_sort]),
                               displayed[np.ix_(iz_sort, ip_sort)], shading="flat", cmap="viridis")
    ax_heat.axhline(focus[focus_index], color="white", ls="--", lw=1)
    ax_heat.set(xlabel="OPD = 2 x DM stroke [um]", ylabel="Absolute mount Z [pulses]",
                title="Measured focus versus OPD")
    fig.colorbar(image, ax=ax_heat, label="Flux / mean at each focus" if normalize_rows else "ADU/frame")
    y = flux[focus_index, segment]
    result = fit_results[focus_index, segment]
    dense = np.linspace(opd.min(), opd.max(), 500)
    ax_trace.plot(opd, y, "o", ms=4, label="Measured")
    ax_trace.plot(dense, evaluate_fit(result, dense), color="tab:orange",
                  ls="-" if result["fit_success"] else "--", label="Fit / candidate")
    ax_trace.set(ylabel="Integrated flux [ADU/frame]", title=f"Z={focus[focus_index]:g}: {result['fit_reason']}")
    ax_trace.legend(fontsize=8)
    ax_trace.text(.02, .04, f"V={result['visibility']:.4g}; R^2={result['r2']:.3g}\n"
                  f"Period={result['period_um']:.3g} um; frequency at bound={result['frequency_at_bound']}",
                  transform=ax_trace.transAxes, fontsize=8)
    ax_resid.axhline(0, color="grey", lw=.8)
    ax_resid.plot(opd, y-evaluate_fit(result, opd), "o", ms=3)
    ax_resid.set(xlabel="OPD = 2 x DM stroke [um]", ylabel="Residual")
    ax_v.plot(focus[iz_sort], visibility[iz_sort, segment], "o-", color="tab:purple")
    ax_v.axvline(focus[focus_index], color="grey", ls="--")
    ax_v.set(xlabel="Absolute mount Z [pulses]", ylabel="Fitted visibility", ylim=(0, MAX_VISIBILITY*1.05),
             title="Visibility versus focus")
    ax_v.grid(alpha=.25)
    fig.suptitle(f"DM segment {segment} - {source_label}", fontsize=13)
    return fig


def inspect_segment(segment=SELECTED_SEGMENT, focus_index=SELECTED_FOCUS_INDEX,
                    normalize_rows=NORMALIZE_FOCUS_OPD_ROWS):
    fig = plot_segment_diagnostics(segment, focus_index, normalize_rows)
    display(fig)
    plt.close(fig)


try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets unavailable; use inspect_segment(segment=0, focus_index=0).")
    inspect_segment()
else:
    controls = dict(segment=widgets.Dropdown(options=list(range(37)), value=SELECTED_SEGMENT, description="Segment"),
                    focus_index=widgets.Dropdown(options=[(f"{i}: Z={z:g}", i) for i, z in enumerate(focus)],
                                                 value=SELECTED_FOCUS_INDEX, description="Focus"),
                    normalize_rows=widgets.Checkbox(value=NORMALIZE_FOCUS_OPD_ROWS, description="Normalize map rows"))
    inspector = widgets.interactive_output(inspect_segment, controls)
    display(widgets.VBox([widgets.HBox(list(controls.values())), inspector]))
    # A static preview also works in notebook frontends without live widget output.
    inspect_segment()

if SAVE_SEGMENT_PNGS:
    for segment in range(37):
        fig = plot_segment_diagnostics(segment)
        fig.savefig(output_dir/f"segment_{segment:02d}_focus_opd.png", dpi=130)
        plt.close(fig)
    print("Saved all 37 per-segment focus/OPD diagnostics.")


## MP4 export
One numbered visibility DM map per focus, in acquisition order, with a common colour
scale. `imageio-ffmpeg` supplies an encoder if system FFmpeg is unavailable. Set
`EXPORT_MP4=False` to skip. The filename contains `synthetic` in preview mode.


In [ ]:
def export_visibility_movie(path):
    executable = shutil.which("ffmpeg")
    if executable is None:
        try:
            import imageio_ffmpeg
            executable = imageio_ffmpeg.get_ffmpeg_exe()
        except (ImportError, RuntimeError) as error:
            raise RuntimeError("MP4 export needs FFmpeg on PATH or the imageio-ffmpeg package") from error
    if MOVIE_FPS <= 0 or MOVIE_HOLD_FRAMES < 1 or int(MOVIE_HOLD_FRAMES) != MOVIE_HOLD_FRAMES:
        raise ValueError("Set positive MOVIE_FPS and positive integer MOVIE_HOLD_FRAMES")
    writer = FFMpegWriter(fps=MOVIE_FPS, codec="libx264", extra_args=["-pix_fmt", "yuv420p"],
                          metadata={"title": f"Fiber DM visibility - {source_label}"})
    # Reuse one figure/canvas for consistent dimensions across every focus frame.
    fig = draw_dm_map(0)
    try:
        with plt.rc_context({"animation.ffmpeg_path": executable}):
            with writer.saving(fig, str(path), dpi=120):
                ax = fig.axes[0]
                collection = ax.collections[0]
                for iz in range(len(focus)):
                    values = visibility[iz]
                    if VISIBILITY_LOG_SCALE:
                        values = np.where(np.isfinite(values), np.maximum(values, VISIBILITY_LIMITS[0]), np.nan)
                    collection.set_array(np.ma.masked_invalid(values))
                    ax.set_title(f"Single-fiber fringe visibility - Z = {focus[iz]:g} pulses\n"
                                 f"{np.isfinite(visibility[iz]).sum()}/37 valid; grey = missing/rejected", fontsize=12)
                    for _ in range(int(MOVIE_HOLD_FRAMES)):
                        writer.grab_frame(facecolor="white")
    finally:
        plt.close(fig)
    return path


if EXPORT_MP4:
    movie_path = output_dir/("synthetic_visibility_vs_focus.mp4" if USE_SYNTHETIC_DEMO else "visibility_vs_focus.mp4")
    export_visibility_movie(movie_path)
    print("Saved:", movie_path)
    display(Video(str(movie_path), embed=True))
